# Coffee shop sales: 20 profiles counted as 30 shops

**Multiple Regression** · a worked example from the StatNex teaching guide

**The story.** A coffee chain has 20 shop profiles: monthly sales, foot traffic, seats and social-media followers. A Frequency column says how many shops share each profile (some rows stand for 2 or 3 shops), so the data describe 30 shops. Head office wants to know which of these measures actually drives monthly sales.

**The question.** Does counting some shops two or three times change which variables the model keeps?

**To run it:**

1. Choose **Copy to Drive** at the top (or **File → Save a copy in Drive**). Your copy opens in a new tab. It is yours, so Colab runs it without a warning, and it keeps your changes.
2. In your copy, choose **Runtime → Run all**.

You can also run this page without a copy. Colab then warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `09_multivariate/multiple_regression__frequency.csv` (20 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
No.,Monthly sales ($000),Foot traffic (thousands/month),Seats,Social followers (thousands),Frequency
1.0,42.3,6.0,39.0,4.7,1.0
2.0,39.3,8.7,11.0,6.5,1.0
3.0,21.2,5.0,11.0,0.3,1.0
4.0,54.2,11.6,46.0,3.7,1.0
5.0,36.4,10.5,14.0,3.7,1.0
6.0,54.5,15.5,49.0,12.8,1.0
7.0,44.3,10.7,44.0,6.2,1.0
8.0,37.2,5.0,45.0,5.2,1.0
9.0,37.7,6.5,27.0,6.9,1.0
10.0,37.4,6.8,40.0,1.6,2.0
11.0,47.8,10.7,42.0,6.6,2.0
12.0,37.4,9.9,28.0,3.1,2.0
13.0,62.3,15.8,48.0,8.4,2.0
14.0,57.9,13.5,51.0,3.0,3.0
15.0,61.1,12.5,51.0,11.4,3.0
16.0,44.8,8.1,49.0,1.5,3.0
17.0,52.9,14.0,10.0,6.1,1.0
18.0,65.6,14.7,58.0,8.7,1.0
19.0,38.4,7.9,19.0,2.7,1.0
20.0,39.0,9.3,16.0,4.0,1.0

These cells save `stepwise_regression.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile stepwise_regression.py
"""Stepwise variable selection for a least-squares regression.

statsmodels has no stepwise procedure, so the rule is written out here. Every step
looks at the model so far and makes at most one move:

* enter -- among the variables not yet in the model, the one with the smallest
  P to enter (or the largest F to enter), if it is below the entry threshold
  (above it, for F);
* remove -- only when nothing entered: among the variables in the model, the one
  with the largest P to remove (smallest F), if it is above the removal threshold
  (below it, for F).

The selection stops when a step makes no move, or when it would return to a set of
variables it has already been through.

F to enter is the square of the t value the variable would have once added, on
1 and n - k - 2 degrees of freedom (k = variables already in). F to remove is the
square of the variable's t value in the current model, on 1 and n - k - 1.
"""

import numpy as np
import statsmodels.api as sm
from scipy import stats


def ols(data, response, names):
    """Least squares of `response` on `names` plus a constant (constant only if empty)."""
    y = data[response].to_numpy(float)
    if names:
        design = sm.add_constant(data[names].to_numpy(float), has_constant="add")
    else:
        design = np.ones((len(y), 1))
    return sm.OLS(y, design).fit()


def f_to_enter(data, response, current, name):
    """F, P and the error df for adding `name` to the variables in `current`."""
    model = ols(data, response, [*current, name])
    f = float(model.tvalues[-1]) ** 2
    df2 = len(data) - len(current) - 2
    return f, float(stats.f.sf(f, 1, df2)), df2


def stepwise(data, response, pool, start, may_enter, may_remove, use_p, enter, remove):
    """Run the selection; one record per step: the model before it, and its move."""
    n = len(data)
    current = list(start)
    steps, seen = [], set()
    while tuple(current) not in seen:
        seen.add(tuple(current))
        entered = removed = None
        if may_enter:
            best = None
            for name in pool:
                if name in current or n - len(current) - 2 < 1:
                    continue
                f, p, _ = f_to_enter(data, response, current, name)
                qualifies = p < enter if use_p else f > enter
                score = p if use_p else -f
                if qualifies and (best is None or score < best[0]):
                    best = (score, name)
            entered = best[1] if best else None
        if may_remove and entered is None and current:
            model = ols(data, response, current)
            df2 = n - len(current) - 1
            worst = None
            for position, name in enumerate(current, start=1):
                f = float(model.tvalues[position]) ** 2
                p = float(stats.f.sf(f, 1, df2))
                qualifies = p > remove if use_p else f < remove
                score = -p if use_p else f
                if qualifies and (worst is None or score < worst[0]):
                    worst = (score, name)
            removed = worst[1] if worst else None
        steps.append({"before": list(current), "entered": entered, "removed": removed})
        if entered is None and removed is None:
            break
        if entered is not None:
            current.append(entered)
        if removed is not None:
            current.remove(removed)
    return steps, current

## Libraries

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from stepwise_regression import f_to_enter, stepwise

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

## The data

In [ ]:
# Text in a numeric column counts as missing, as it does in the app.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
response = "Monthly sales ($000)"
predictors = ["Foot traffic (thousands/month)", "Seats", "Social followers (thousands)"]
reference = []
columns = [*predictors, *reference, response]
frame = data[columns].apply(pd.to_numeric, errors="coerce")
confidence = 95 / 100

## Frequency

In [ ]:
# Each row stands for as many cases as its frequency, so the row is repeated that many
# times. Weighted least squares would give the same coefficients but would count the
# error degrees of freedom in rows rather than cases, and every SE and P with them.
weights = pd.to_numeric(data["Frequency"], errors="coerce")
usable = weights.notna() & (weights > 0)
frequency_summary = pd.DataFrame(
    {"n": [usable.sum(), (~usable).sum(), len(frame)], "%": [usable.mean(), (~usable).mean(), 1.0]},
    index=["usable rows", "unusable rows", "total"],
)
print(frequency_summary)
kept = frame[usable]
frame = kept.loc[kept.index.repeat(weights[usable].round().astype(int))].reset_index(drop=True)

## Cases

In [ ]:
# A case is used only when the response and every explanatory variable are present.
y_ok = frame[response].notna()
x_ok = frame[predictors].notna().all(axis=1)
valid = frame[y_ok & x_ok]
n = len(valid)
summary = pd.DataFrame(
    {"n": [(y_ok & x_ok).sum(), (~y_ok & x_ok).sum(), (y_ok & ~x_ok).sum(),
           (~y_ok & ~x_ok).sum(), len(frame)]},
    index=["valid", "response missing", "explanatory missing", "both missing", "total"],
)
summary["%"] = summary["n"] / len(frame)
print(summary)

## Descriptive statistics and correlation matrix

In [ ]:
# The correlation matrix shows each pair's straight-line link on its own, before the
# regression holds the other variables fixed.
descriptive = pd.DataFrame({
    "n": n, "mean": valid[columns].mean(), "variance": valid[columns].var(ddof=1),
    "SD": valid[columns].std(ddof=1), "min": valid[columns].min(), "max": valid[columns].max(),
})
correlation = valid[columns].corr()
print()
print(descriptive)
print()
print(correlation)


# Least squares of the response on `names` plus a constant (the constant alone when
# `names` is empty).
def fit(names):
    y = valid[response].to_numpy(float)
    if not names:
        return sm.OLS(y, np.ones((len(y), 1))).fit()
    return sm.OLS(y, sm.add_constant(valid[names].to_numpy(float), has_constant="add")).fit()


# R, adjusted R, R², adjusted R², the Durbin-Watson ratio and AIC. R² is the share of
# the response's variation the model explains; adjusted R² charges for each variable,
# so it can fall when a useless one is added. Durbin-Watson compares each residual
# with the next one (near 2: no pattern from row to row); it means something only
# when the rows have an order, such as time. AIC: lower is better, for models of the
# same response on the same cases. Here it is n·ln(SSe/n) + 2(k + 1), the textbook
# regression form, as R's extractAIC() gives it. statsmodels' model.aic uses the full
# normal log-likelihood and comes out larger by n·(ln 2π + 1) for every model, so the
# two rank models alike. With no variable in the model the adjusted values and AIC
# are not printed.
def accuracy(model, k):
    if k == 0:
        return [0.0, np.nan, 0.0, np.nan, durbin_watson(model.resid), np.nan]
    return [np.sqrt(model.rsquared), np.sqrt(max(model.rsquared_adj, 0.0)), model.rsquared,
            model.rsquared_adj, durbin_watson(model.resid),
            n * np.log(model.ssr / n) + 2 * (k + 1)]


# The analysis of variance: the response's variation about its mean (total) split
# into the part the model explains (regression) and the part it leaves (error). F is
# the regression mean square over the error mean square; its P value tests whether
# all the coefficients are 0 at once.
def anova_rows(model, k):
    ss_total, ss_error = model.centered_tss, model.ssr
    regression = [ss_total - ss_error, k, (ss_total - ss_error) / k, model.fvalue,
                  model.f_pvalue] if k else [0.0, 0, np.nan, np.nan, np.nan]
    return [regression,
            [ss_error, n - k - 1, ss_error / (n - k - 1), np.nan, np.nan],
            [ss_total, n - 1, np.nan, np.nan, np.nan]]


# One row per variable, then the constant. For each variable: the coefficient b (the
# change in the response per unit of the variable, the others held fixed), its
# standard error, the standardised coefficient b·SD(x)/SD(y) (comparable across
# variables), the confidence interval from conf_int(), F = t², t = b/SE and its P
# value (H0: b = 0), the simple correlation with the response, the partial correlation
# from t and the error df, and the tolerance and VIF. The VIF (variance inflation
# factor) says how much the coefficient's variance is inflated because its variable
# overlaps with the others: 1 is none, and above 10 is the usual rule-of-thumb
# warning; tolerance is 1/VIF.
def coefficient_table(model, names):
    k = len(names)
    interval = model.conf_int(alpha=1 - confidence)
    exog = model.model.exog
    rows = []
    for j, name in enumerate(names, start=1):
        b, se, t = model.params[j], model.bse[j], model.tvalues[j]
        vif = variance_inflation_factor(exog, j)
        rows.append([b, se, b * valid[name].std() / valid[response].std(), *interval[j],
                     t**2, t, model.pvalues[j],
                     valid[response].corr(valid[name]), t / np.sqrt(t**2 + n - k - 1),
                     1 / vif, vif])
    b, se, t = model.params[0], model.bse[0], model.tvalues[0]
    rows.append([b, se, np.nan, *interval[0], t**2, t, model.pvalues[0],
                 np.nan, np.nan, np.nan, np.nan])
    return pd.DataFrame(rows, index=[*names, "Constant"],
                        columns=["b", "SE", "beta", "lower", "upper", "F", "t", "P",
                                 "simple r", "partial r", "tolerance", "VIF"])

## Variable selection

In [ ]:
# statsmodels has no stepwise selection, so the app's rule -- enter and remove
# variables by their P (or F) value -- is written out in stepwise_regression.py beside
# this file. A P value from the final model of a selection is smaller than it would
# be for a variable chosen in advance, because the variables were picked for their
# small P values: read it as a screening result, not a confirmed test.
# Forward stepwise: start with no variable. Each step enters the best variable that
# qualifies; only when none does, it removes the worst one that qualifies for removal.
start, may_enter, may_remove = [], True, True
# Criterion: P value. Enter below the entry value, remove above the removal value.
use_p = True
enter_threshold, remove_threshold = 0.2, 0.2
steps, selected = stepwise(valid, response, predictors, start, may_enter, may_remove,
                           use_p, enter_threshold, remove_threshold)
print(f"\nSelected: {selected}")

## The selection, step by step

In [ ]:
# Each step shows the model before its move, and which variable the move enters or
# removes. `excluded` lists, for each variable still out, its F and P to enter: the
# test of adding it to that step's model.
selection_settings = [enter_threshold, remove_threshold]
process_rows, step_anova_rows, step_coefficients, excluded_rows = [], [], [], []
for number, step in enumerate(steps):
    names = step["before"]
    model = fit(names)
    process_rows.append(accuracy(model, len(names)))
    step_anova_rows += anova_rows(model, len(names))
    step_coefficients.append(coefficient_table(model, names).assign(step=number))
    for name in predictors:
        if name not in names:
            excluded_rows.append([number, name, *f_to_enter(valid, response, names, name)])
process = pd.DataFrame(process_rows, columns=["R", "adjusted R", "R2", "adjusted R2", "DW", "AIC"])
process["entered"] = [step["entered"] for step in steps]
process["removed"] = [step["removed"] for step in steps]
step_anova = pd.DataFrame(step_anova_rows, columns=["SS", "df", "MS", "F", "P"])
step_anova.insert(0, "source", ["regression", "error", "total"] * len(steps))
step_anova.insert(0, "step", np.repeat(range(len(steps)), 3))
step_coefficients = pd.concat(step_coefficients)
excluded = pd.DataFrame(excluded_rows, columns=["step", "variable", "F", "P", "df2"])
excluded.insert(3, "df1", 1)
excluded = excluded[["step", "variable", "F", "df1", "df2", "P"]]
print()
print(process)
print()
print(step_anova.to_string(index=False))
print()
print(step_coefficients)
print()
print(excluded.to_string(index=False))

## The model

In [ ]:
# model.summary() would print statsmodels' own report of the same fit.
model = fit(selected)
k = len(selected)
fit_accuracy = pd.DataFrame([accuracy(model, k)],
                            columns=["R", "adjusted R", "R2", "adjusted R2", "DW", "AIC"])
anova = pd.DataFrame(anova_rows(model, k), index=["regression", "error", "total"],
                     columns=["SS", "df", "MS", "F", "P"])
coefficients = coefficient_table(model, selected)
print()
print(fit_accuracy.to_string(index=False))
print()
print(anova)
print()
print(coefficients)

## Predicted values, residuals and diagnostics

In [ ]:
# For each case: the predicted value, the residual (observed - predicted), the
# standardised residual (the residual over the residual SD), the studentised residual,
# which also allows for the case's leverage (beyond about +-2 is worth a look), Cook's
# distance (how much the whole fit moves if the case is dropped) and the leverage
# (how unusual the case's explanatory values are).
influence = model.get_influence()
predictions = pd.DataFrame({
    "no": np.arange(1, n + 1), "observed": valid[response].to_numpy(float),
    "predicted": model.fittedvalues, "residual": model.resid,
    "standardised": model.resid / np.sqrt(model.mse_resid),
    "studentised": influence.resid_studentized_internal,
    "Cook": influence.cooks_distance[0], "leverage": influence.hat_matrix_diag,
})
print()
print(predictions.to_string(index=False))

## Tests for heteroscedasticity

In [ ]:
# Regression assumes the residuals spread equally at every value of the explanatory
# variables. These tests ask whether the squared residuals rise or fall with them; a
# small P is a warning that the standard errors and P values above may be off (a
# large one does not prove the spread is equal).
# Breusch-Pagan in Koenker's form, het_breuschpagan()'s default, which does not assume
# normal residuals: n·R² of the squared residuals regressed on the variables.
# White's adds their squares and cross-products, k + k(k+1)/2 terms -- fewer when a
# product repeats a column, as a 0/1 variable's square does, so its df is the rank of
# that design less 1, the df het_white's P is computed on.
exog = model.model.exog
bp = het_breuschpagan(model.resid, exog)
white = het_white(model.resid, exog)
upper = np.triu_indices(exog.shape[1])
white_df = np.linalg.matrix_rank(exog[:, upper[0]] * exog[:, upper[1]]) - 1
hetero = pd.DataFrame([[bp[0], k, bp[1]], [white[0], white_df, white[1]]],
                      index=["Breusch-Pagan", "White"], columns=["chi-square", "df", "P"])
print()
print(hetero)

## Normality of the residuals

In [ ]:
# The t and F tests assume normally distributed errors. Shapiro-Wilk on the final
# model's residuals: W near 1 and a large P mean no clear departure from normality
# (not proof of it). SciPy and R's shapiro.test both run Royston's algorithm AS R94,
# which covers 3 to 5,000 cases.
if 3 <= len(model.resid) <= 5000:
    shapiro_w, shapiro_p = stats.shapiro(model.resid)
    normality = [shapiro_w, len(model.resid), shapiro_p]
    print(f"\nShapiro-Wilk W = {shapiro_w:.4f}, n = {len(model.resid)}, P = {shapiro_p:.4f}")

## What the result says

This run uses the same 20 shop profiles, but a frequency column says how many shops share each profile, so the regression counts 30 shops. The method is again forward stepwise selection: a predictor enters while its P value, the chance of a link this strong if it added nothing, is below 0.2.

Now all three predictors enter. Social followers come in at step 3 with P = 0.073. R squared, the share of the differences in sales that the model accounts for, is 0.90.

But look at the coefficient of social followers: 0.46, with a confidence interval (the range of values the data fit) from -0.05 to 0.96. That range still includes zero, which means no link. The repeated rows add no new shops, only more confidence. So yes, the counting changes which variables the model keeps. It does not show that followers drive sales.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.